# Data Exploration: ATIS and SNIPS Datasets

This notebook explores the standard NLU benchmark datasets used for
intent classification and slot filling:

- **ATIS**: Airline Travel Information System (Hemphill et al., 1990)
- **SNIPS**: Voice assistant queries (Coucke et al., 2018)

We analyze intent distributions, slot type frequencies, sequence lengths,
and label co-occurrence patterns to inform modeling decisions.

In [ ]:
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def load_split(data_dir: Path) -> dict:
    """Load a dataset split from the standard seq.in/seq.out/label format."""
    utterances = (data_dir / "seq.in").read_text().strip().splitlines()
    slot_seqs = (data_dir / "seq.out").read_text().strip().splitlines()
    intents = (data_dir / "label").read_text().strip().splitlines()
    return {
        "utterances": utterances,
        "slot_seqs": slot_seqs,
        "intents": intents,
    }

## Load Data

Download ATIS and SNIPS datasets into `data/atis/` and `data/snips/`
before running this notebook. Each split should contain:
- `seq.in`: one utterance per line (space-separated tokens)
- `seq.out`: one slot label sequence per line (space-separated BIO tags)
- `label`: one intent label per line

In [ ]:
# Update these paths to point to your downloaded datasets
DATA_ROOT = Path("../data")

datasets = {}
for name in ["atis", "snips"]:
    dataset_dir = DATA_ROOT / name
    if dataset_dir.exists():
        datasets[name] = {
            split: load_split(dataset_dir / split)
            for split in ["train", "test"]
            if (dataset_dir / split).exists()
        }
        print(f"Loaded {name}: {', '.join(f'{k}={len(v["utterances"])}' for k, v in datasets[name].items())}")
    else:
        print(f"Dataset not found: {dataset_dir}")

## Intent Distribution

ATIS has a heavily skewed intent distribution (dominated by `atis_flight`),
while SNIPS is more balanced across its 7 intents.

In [ ]:
for name, splits in datasets.items():
    if "train" not in splits:
        continue
    intent_counts = Counter(splits["train"]["intents"])
    
    print(f"\n{name.upper()} Intent Distribution (train):")
    print(f"  Total intents: {len(intent_counts)}")
    print(f"  Total examples: {sum(intent_counts.values())}")
    print(f"  Top 5:")
    for intent, count in intent_counts.most_common(5):
        pct = count / sum(intent_counts.values()) * 100
        print(f"    {intent}: {count} ({pct:.1f}%)")

## Sequence Length Analysis

Understanding sequence lengths helps set the `max_seq_length` hyperparameter.
Too short truncates information; too long wastes compute on padding.

In [ ]:
for name, splits in datasets.items():
    if "train" not in splits:
        continue
    lengths = [len(u.split()) for u in splits["train"]["utterances"]]
    
    print(f"\n{name.upper()} Sequence Lengths:")
    print(f"  Mean: {np.mean(lengths):.1f}")
    print(f"  Median: {np.median(lengths):.1f}")
    print(f"  Max: {max(lengths)}")
    print(f"  95th percentile: {np.percentile(lengths, 95):.0f}")
    print(f"  99th percentile: {np.percentile(lengths, 99):.0f}")

## Slot Type Analysis

Count the frequency of each slot type (ignoring BIO prefixes) to understand
which entities are most common in each dataset.

In [ ]:
for name, splits in datasets.items():
    if "train" not in splits:
        continue
    
    slot_counts = Counter()
    for seq in splits["train"]["slot_seqs"]:
        for label in seq.split():
            if label != "O":
                # Strip B-/I- prefix to get slot type
                slot_type = label.split("-", 1)[1] if "-" in label else label
                slot_counts[slot_type] += 1
    
    print(f"\n{name.upper()} Slot Types: {len(slot_counts)} unique types")
    print(f"  Top 10:")
    for slot, count in slot_counts.most_common(10):
        print(f"    {slot}: {count}")